Get the following for more data collection:
- Time in epoch of next market open
- Time in epoch of last market close

Get the following for training
- DOW of article
- Time of day of article in EST
- Time to next open in epoch
- Time from last close in epoch

Tiingo for training at time of article
- Market cap
- Enterprise Val
- Pe ratio
- Pb ratio
- trailingPEG1y

Polygon for training
- Sic code
- Price at last close. Price a week, month, year ago.
- Last day vol, avg daily from last week, month, year volume
- Last day trans, avg daily trans from last week, month, year
- Average bid ask spread in last day, week, month, year taking 12 samples from each
- Next three full day vol

Polygon for results
- Bid, Ask, and last price at open and every 5 minutes till 1000, every hour til close 



In [61]:
my_datetime = datetime(2026, 1, 5, 9, 33)
et_tz = pytz.timezone("America/New_York")
et_datetime = et_tz.localize(my_datetime) if my_datetime.tzinfo is None else my_datetime.astimezone(et_tz)

print(is_market_open(et_datetime))
print(previous_year_force(et_datetime))
print(et_datetime)
print(get_market_holidays(2026))

True
2025-01-06 09:33:00-05:00
2026-01-05 09:33:00-05:00
{datetime.date(2026, 2, 16), datetime.date(2026, 6, 19), datetime.date(2026, 11, 26), datetime.date(2026, 1, 19), datetime.date(2026, 12, 25), datetime.date(2026, 9, 7), datetime.date(2026, 1, 1), datetime.date(2026, 5, 25), datetime.date(2026, 4, 3), datetime.date(2026, 7, 3)}


In [2]:
from datetime import datetime, timedelta
import pytz
from dateutil.easter import easter as du_easter

def get_market_holidays(year):
    """ Returns a set of actual NYSE market holidays for a given year, dynamically calculated. """
    
    def nth_weekday(n, weekday, month):
        """ Returns the date of the nth occurrence of a specific weekday in a given month. """
        first_day = datetime(year, month, 1)
        first_occurrence = first_day + timedelta(days=(weekday - first_day.weekday() + 7) % 7)
        return first_occurrence + timedelta(weeks=n-1)
    
    def last_weekday(weekday, month):
        """ Returns the date of the last occurrence of a specific weekday in a given month. """
        last_day = datetime(year, month + 1, 1) - timedelta(days=1)
        while last_day.weekday() != weekday:
            last_day -= timedelta(days=1)
        return last_day

    # Dynamically calculated holidays
    mlk_day = nth_weekday(3, 0, 1)       # Martin Luther King Jr. Day (3rd Monday of January)
    presidents_day = nth_weekday(3, 0, 2) # Presidents' Day (3rd Monday of February)
    memorial_day = last_weekday(0, 5)     # Memorial Day (Last Monday of May)
    labor_day = nth_weekday(1, 0, 9)      # Labor Day (1st Monday of September)
    thanksgiving = nth_weekday(4, 3, 11)  # Thanksgiving (4th Thursday of November)

    

    # Fixed holidays
    fixed_holidays = {
        datetime(year, 1, 1),  # New Year's Day
        datetime(year, 7, 4),  # Independence Day
        datetime(year, 12, 25), # Christmas Day
        datetime(year, 6, 19)
    }

    # Full holiday set
    market_holidays = {mlk_day, presidents_day, memorial_day, labor_day, thanksgiving, 
                      } | fixed_holidays

    # Weekend adjustment (if holiday falls on a weekend, observe on nearest weekday)
    observed_holidays = set()
    for holiday in market_holidays:
        if holiday.weekday() == 5:  # Saturday -> Observed on Friday
            observed_holidays.add((holiday - timedelta(days=1)).date())
        elif holiday.weekday() == 6:  # Sunday -> Observed on Monday
            observed_holidays.add((holiday + timedelta(days=1)).date())
        else:
            observed_holidays.add(holiday.date())
    # Good Friday is two days before Easter Sunday
    good_friday = du_easter(year) - timedelta(days=2)
    observed_holidays.add(good_friday)

    return observed_holidays


def is_market_open(et_datetime):
    """ Returns True if the given ET datetime is during market hours (9:30 AM - 4:00 PM ET) on a trading day. """
    market_open_hour, market_open_minute = 9, 30
    market_close_hour = 16  # 4:00 PM ET
    et_date = et_datetime.date()

    if et_datetime.weekday() >= 5 or et_date in get_market_holidays(et_datetime.year):
        return False  # Market closed on weekends and holidays

    return (market_open_hour < et_datetime.hour < market_close_hour) or (
        et_datetime.hour == market_open_hour and et_datetime.minute >= market_open_minute
    )


def previous_trading_day(dt):
    """ Returns the last market close date, skipping weekends and holidays. """
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt -= timedelta(days=1)
    return dt

def previous_trading_day_force(dt):
    """ Returns the last market close date, skipping weekends and holidays. """
    dt -= timedelta(days=1)
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt -= timedelta(days=1)
    return dt

def next_trading_day(dt):
    """ Returns the next market open date, skipping weekends and holidays. """
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt += timedelta(days=1)
    return dt

def next_trading_day_force(dt):
    """ Returns the next market open date, skipping weekends and holidays. """
    dt += timedelta(days=1)
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt += timedelta(days=1)
    return dt

def previous_week_force(dt):
    dt -= timedelta(weeks=1)
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt -= timedelta(days=1)
    return dt

def previous_month_force(dt):
    dt -= timedelta(weeks=4)
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt -= timedelta(days=1)
    return dt

def previous_year_force(dt):
    dt -= timedelta(weeks=52)
    while dt.weekday() >= 5 or dt.date() in get_market_holidays(dt.year):
        dt -= timedelta(weeks=52)
    return dt

def seconds_to_next_market_open(et_datetime):
    """ Calculates seconds until the next market open (9:30 AM ET) considering holidays. Returns 0 if market is open. """
    market_open_hour = 9
    market_open_minute = 30
    et_tz = pytz.timezone("America/New_York")

    # Convert to aware ET datetime
    et_datetime = et_tz.localize(et_datetime) if et_datetime.tzinfo is None else et_datetime.astimezone(et_tz)

    # If market is open, return 0
    if is_market_open(et_datetime):
        return 0

    # Find the next trading day
    next_day = next_trading_day(et_datetime + timedelta(days=1)) if et_datetime.hour >= market_open_hour else et_datetime
    if next_day.date() in get_market_holidays(next_day.year):
        next_day = next_trading_day(next_day + timedelta(days=1))

    # Set market open time for that day
    next_open = next_day.replace(hour=market_open_hour, minute=market_open_minute, second=0, microsecond=0)

    # Calculate seconds difference
    return max(0, int((next_open - et_datetime).total_seconds()))

def seconds_from_last_market_close(et_datetime):
    """ Calculates seconds since the last market close (4:00 PM ET) considering holidays. Returns 0 if market is open. """
    market_close_hour = 16  # 4:00 PM ET
    et_tz = pytz.timezone("America/New_York")

    # Convert to aware ET datetime
    et_datetime = et_tz.localize(et_datetime) if et_datetime.tzinfo is None else et_datetime.astimezone(et_tz)

    # If market is open, return 0
    if is_market_open(et_datetime):
        return 0

    # Find the previous trading day
    prev_day = previous_trading_day(et_datetime - timedelta(days=1)) if et_datetime.hour < market_close_hour else et_datetime
    if prev_day.date() in get_market_holidays(prev_day.year):
        prev_day = previous_trading_day(prev_day - timedelta(days=1))

    # Set market close time for that day
    last_close = prev_day.replace(hour=market_close_hour, minute=0, second=0, microsecond=0)

    # Calculate seconds difference
    return max(0, int((et_datetime - last_close).total_seconds()))


In [39]:
import csv
from datetime import datetime
import calendar
import requests
import time
import ast
from polygon.rest import RESTClient

client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")

NS_PER_MIN = 60000000000
NS_PER_DAY = NS_PER_MIN * 60 * 24
DAILY_AVG_NS = [60*NS_PER_MIN, 180*NS_PER_MIN, 300*NS_PER_MIN]
DAY_OF_RESULTS = [1*NS_PER_MIN, 3*NS_PER_MIN, 5*NS_PER_MIN, 10*NS_PER_MIN, 30*NS_PER_MIN, 90*NS_PER_MIN, 150*NS_PER_MIN, 210*NS_PER_MIN, 270*NS_PER_MIN, 330*NS_PER_MIN]
NEXT_DAY_RESULTS = [1*NS_PER_MIN, 30*NS_PER_MIN]

def get_bid_ask(ticker, stamp):
    url = 'https://api.polygon.io/v3/quotes/' + ticker + "?timestamp.gte=" + str(stamp) + "&order=asc&limit=1&sort=timestamp&apiKey=iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk"
    res = requests.get(url)
    res = res.json()["results"][0]
    bid = res["bid_price"]
    ask = res["ask_price"]
    spread = ask - bid
    return bid, ask, spread

with open("sic_dic.txt", "r") as f:
    content = f.read().strip()
    sic_dic = ast.literal_eval(content)


#INPUT
#ticker,timestamp,category,ST_impact_sentiment,text

#OUTPUT
#ticker,timestamp,category,impact,sentimet,DOW,timeESTInSecondsAfter4PM,secToNextOpen
#secFromLastClose, marketCap, enterpriseVal, PERaito, PBRatio, trailingPEG1Y
#first2Sic, first3Sic, first4Sic
#priceLastClose, priceLast2DayClose, priceLastWeekClose, priceLastMonthClose, priceLast3MonthClose, priceLastYearClose
#volLastCloseAvg, volLast2DayCloseAvg, volLastWeekCloseAvg, volLastMonthCloseAvg, volLast3MonthCloseAvg, volLastYearCloseAvg
#transLastClose, transLast2DayClose, transLastWeekClose, transLastMonthClose, transLast3MonthClose, transLastYearClose
#volNextDay, volNext2Day, volNext3Day
#priceLast1030, priceLast1230, priceLast230, avgLastDaySpread, avgLastWeekSpread, avgLastMonthSpread
#nextBid931, nextAsk931, nextBid933, nextAsk933, nextBid935, nextAsk935, nextBid940, nextAsk940, nextBid1000, nextAsk1000, nextBid1100, nextAsk1100, nextBid1200, nextAsk1200, nextBid1300, nextAsk1300, nextBid1400, nextAsk1400, nextBid1500, nextAsk1500
#nextNextBid931, nextNextAsk931, nextNextBid1000, nextNextAsk1000

with open('CleanedTitanData.csv', newline='') as csvfile:
  quoter = list(csv.reader(csvfile, delimiter=','))
  for i in range(100):#range(len(quoter)):
    try:
        row = quoter[i]
        buffer = row[0:3]
        ticker = row[0]
        time_string = row[1]
        time_datetime = datetime.fromisoformat(time_string)
        buffer.append(int(str(row[3])[0]))
        buffer.append(int(str(row[3])[-1]))
        buffer.append(time_datetime.weekday())

        last_16 = time_datetime.replace(hour=16, minute=0, second=0, microsecond=0)
        # If date_obj is before 16:00, go back to the previous day's 16:00
        if time_datetime < last_16:
            last_16 -= timedelta(days=1)
        # Calculate the seconds difference
        seconds_since_16 = int((time_datetime - last_16).total_seconds())
        buffer.append(seconds_since_16)
        
        buffer.append(int(seconds_to_next_market_open(time_datetime)))
        buffer.append(int(seconds_from_last_market_close(time_datetime)))

        headers = {
        'Content-Type': 'application/json',
        }
        endDate = time_string[:10]
        startDate = str(time_datetime - timedelta(days=4))[:10]
        url_string = "https://api.tiingo.com/tiingo/fundamentals/" + ticker.lower() + "/daily?token=c7dd3b16d9e5b5a440522b13fc9b527e45b07411"
        requestResponse = requests.get(url_string, headers=headers, 
        params={'startDate' : startDate,
            "endDate" : endDate})
        result = requestResponse.json()[-1]
        buffer.append(result["marketCap"])
        buffer.append(result["enterpriseVal"])
        buffer.append(result["peRatio"])
        buffer.append(result["pbRatio"])
        buffer.append(result["trailingPEG1Y"])

        sic_code = sic_dic[ticker]

        buffer.append(int(str(sic_code)[:2]))
        buffer.append(int(str(sic_code)[:3]))
        buffer.append(int(str(sic_code)))


        last_close_date = time_datetime - timedelta(seconds=seconds_from_last_market_close(time_datetime))
        year_ago_last_close_date = last_close_date - timedelta(days=380)
        year_ago_last_close_date = str(year_ago_last_close_date)[:10]
        last_close_date = str(last_close_date)[:10]

        aggs = []
        for a in client.list_aggs(
            ticker,
            1,
            "day",
            year_ago_last_close_date,
            last_close_date,
            adjusted="true",
            sort="desc",
            limit=250,
        ):
            aggs.append(a)

        if len(aggs) < 253:
            print(ticker + " LESS THAN 253 DAYS")
            continue

        last_close_price = aggs[0].close
        last_2day_close_price = aggs[1].close
        last_week_close_price = aggs[5].close
        last_month_close_price = aggs[22].close
        last_3month_close_price = aggs[66].close
        last_year_close_price = aggs[252].close

        last_close_vol, last_2day_close_vol, last_week_close_vol, last_month_close_vol, last_3month_close_vol, last_year_close_vol = 0,0,0,0,0,0
        running_vol = 0
        for i in range(253):
            running_vol += aggs[i].volume
            if i == 0:
                last_close_vol = running_vol/(i+1)
            elif i == 1:
                last_2day_close_vol = running_vol/(i+1)
            elif i == 5:
                last_week_close_vol = running_vol/(i+1)
            elif i == 22:
                last_month_close_vol = running_vol/(i+1)
            elif i == 66:
                last_3month_close_vol = running_vol/(i+1)
            elif i == 252:
                last_year_close_vol = running_vol/(i+1)

        last_close_trans, last_2day_close_trans, last_week_close_trans, last_month_close_trans, last_3month_close_trans, last_year_close_trans = 0,0,0,0,0,0
        running_trans = 0
        for i in range(253):
            running_trans += aggs[i].transactions
            if i == 0:
                last_close_trans = running_trans/(i+1)
            elif i == 1:
                last_2day_close_trans = running_trans/(i+1)
            elif i == 5:
                last_week_close_trans = running_trans/(i+1)
            elif i == 22:
                last_month_close_trans = running_trans/(i+1)
            elif i == 66:
                last_3month_close_trans = running_trans/(i+1)
            elif i == 252:
                last_year_close_trans = running_trans/(i+1)

        buffer.append(last_close_price)
        buffer.append(last_2day_close_price)
        buffer.append(last_week_close_price)
        buffer.append(last_month_close_price)
        buffer.append(last_3month_close_price)
        buffer.append(last_year_close_price)

        buffer.append(last_close_vol)
        buffer.append(last_2day_close_vol)
        buffer.append(last_week_close_vol)
        buffer.append(last_month_close_vol)
        buffer.append(last_3month_close_vol)
        buffer.append(last_year_close_vol)

        buffer.append(last_close_trans)
        buffer.append(last_2day_close_trans)
        buffer.append(last_week_close_trans)
        buffer.append(last_month_close_trans)
        buffer.append(last_3month_close_trans)
        buffer.append(last_year_close_trans)

        next_open_date = time_datetime + timedelta(seconds=seconds_to_next_market_open(time_datetime))
        next_week_next_open_date = next_open_date + timedelta(days=7)
        next_week_next_open_date = str(next_week_next_open_date)[:10]
        next_open_date = str(next_open_date)[:10]

        aggs = []
        for a in client.list_aggs(
            ticker,
            1,
            "day",
            next_open_date,
            next_week_next_open_date,
            adjusted="true",
            sort="asc",
            limit=5,
        ):
            aggs.append(a)

        next_open_vol = aggs[0].volume
        next_2day_open_vol = aggs[1].volume
        next_3day_open_vol = aggs[2].volume

        buffer.append(next_open_vol)
        buffer.append(next_2day_open_vol)
        buffer.append(next_3day_open_vol)

        last_close_date = time_datetime - timedelta(seconds=seconds_from_last_market_close(time_datetime))
        last_close_week = last_close_date - timedelta(days=11)
        last_close_month  = last_close_date - timedelta(days=33)
        last_close_week = (last_close_week) + timedelta(seconds=seconds_to_next_market_open((last_close_week)))
        last_close_month = (last_close_month)+ timedelta(seconds=seconds_to_next_market_open((last_close_month)))
        last_open_date_ns = (int(last_close_date.timestamp()) * 1000000000) - (NS_PER_MIN*(60*6.5))
        last_open_week_ns = (int(last_close_week.timestamp()) * 1000000000)
        last_open_month_ns = (int(last_close_month.timestamp()) * 1000000000)

        doBreak = False
        running_spread = 0
        for slot in DAILY_AVG_NS:
            timestamp = int(last_open_date_ns + slot)
            bid, ask, spread = get_bid_ask(ticker, timestamp)
            if bid == 0 or ask == 0:
                bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
            if bid == 0 or ask == 0:
                doBreak = True
            buffer.append((bid + ask)/2)
            running_spread += spread
        if doBreak:
            print("BAD QUOTE")
            continue
            
        buffer.append(running_spread/len(DAILY_AVG_NS))

        running_spread = 0
        for i in range(len(DAILY_AVG_NS)):
            timestamp = int(last_open_week_ns + DAILY_AVG_NS[i])
            bid, ask, spread = get_bid_ask(ticker, timestamp)
            if bid == 0 or ask == 0:
                bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
            if bid == 0 or ask == 0:
                doBreak = True
            running_spread += spread
            last_close_week = (last_close_week + timedelta(seconds=60*60*8)) + timedelta(seconds=seconds_to_next_market_open((last_close_week + timedelta(seconds=60*60*8))))
            last_open_week_ns = (int(last_close_week.timestamp()) * 1000000000)
        if doBreak:
            print("BAD QUOTE")
            continue
        buffer.append(running_spread/len(DAILY_AVG_NS))

        running_spread = 0
        for i in range(len(DAILY_AVG_NS)):
            timestamp = int(last_open_month_ns + DAILY_AVG_NS[i])
            bid, ask, spread = get_bid_ask(ticker, timestamp)
            if bid == 0 or ask == 0:
                bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
            if bid == 0 or ask == 0:
                doBreak = True
            running_spread += spread
            last_close_month = (last_close_month + timedelta(seconds=60*60*8))+ timedelta(seconds=seconds_to_next_market_open((last_close_month + timedelta(seconds=60*60*8))))
            last_open_month_ns = (int(last_close_month.timestamp()) * 1000000000)
        if doBreak:
            print("BAD QUOTE")
            continue

        buffer.append(running_spread/len(DAILY_AVG_NS))

        next_open_date = time_datetime + timedelta(seconds=seconds_to_next_market_open(time_datetime))
        next_next_open_date = next_open_date + timedelta(seconds=60*60*8) + timedelta(seconds=seconds_to_next_market_open(next_open_date+timedelta(seconds=60*60*8)))
        next_open_date_ns = int(next_open_date.timestamp()) * 1000000000
        next_next_open_date_ns = int(next_next_open_date.timestamp()) * 1000000000

        for slot in DAY_OF_RESULTS:
            timestamp = next_open_date_ns + slot
            bid, ask, spread = get_bid_ask(ticker, timestamp)
            if bid == 0 or ask == 0:
                bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
            if bid == 0 or ask == 0:
                doBreak = True
            buffer.append(bid)
            buffer.append(ask)
        if doBreak:
            print("BAD QUOTE")
            continue

        for slot in NEXT_DAY_RESULTS:
            timestamp = next_next_open_date_ns + slot
            bid, ask, spread = get_bid_ask(ticker, timestamp)
            if bid == 0 or ask == 0:
                bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
            if bid == 0 or ask == 0:
                doBreak = True
            buffer.append(bid)
            buffer.append(ask)
        if doBreak:
            print("BAD QUOTE")
            continue

        buffer.append(row[4])

        with open('ProcessedTitanData.csv', 'a', newline='') as csvfilewrite:
            #print(quoter[i][0] + " YAY")
            cleaner = csv.writer(csvfilewrite, delimiter=',')
            cleaner.writerow(buffer)
            csvfilewrite.close()
    except:
        print("WOAH FAILED FOR SOME REASON")



    

WOAH FAILED FOR SOME REASON


In [35]:
NS_PER_MIN = 60000000000
NS_PER_DAY = NS_PER_MIN * 60 * 24
DAILY_AVG_NS = [60*NS_PER_MIN, 180*NS_PER_MIN, 300*NS_PER_MIN]
DAY_OF_RESULTS = [1*NS_PER_MIN, 3*NS_PER_MIN, 5*NS_PER_MIN, 10*NS_PER_MIN, 30*NS_PER_MIN, 90*NS_PER_MIN, 150*NS_PER_MIN, 210*NS_PER_MIN, 270*NS_PER_MIN, 330*NS_PER_MIN]
NEXT_DAY_RESULTS = [1*NS_PER_MIN, 30*NS_PER_MIN]

def get_bid_ask(ticker, stamp):
    url = 'https://api.polygon.io/v3/quotes/' + ticker + "?timestamp.gte=" + str(stamp) + "&order=asc&limit=1&sort=timestamp&apiKey=iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk"
    res = requests.get(url)
    res = res.json()["results"][0]
    bid = res["bid_price"]
    ask = res["ask_price"]
    spread = ask - bid
    return bid, ask, spread
buffer = []


last_close_date = time_datetime - timedelta(seconds=seconds_from_last_market_close(time_datetime))
last_close_week = last_close_date - timedelta(days=11)
last_close_month  = last_close_date - timedelta(days=33)
last_close_week = (last_close_week) + timedelta(seconds=seconds_to_next_market_open((last_close_week)))
last_close_month = (last_close_month)+ timedelta(seconds=seconds_to_next_market_open((last_close_month)))
last_open_date_ns = (int(last_close_date.timestamp()) * 1000000000) - (NS_PER_MIN*(60*6.5))
last_open_week_ns = (int(last_close_week.timestamp()) * 1000000000)
last_open_month_ns = (int(last_close_month.timestamp()) * 1000000000)

doBreak = False
running_spread = 0
for slot in DAILY_AVG_NS:
    timestamp = int(last_open_date_ns + slot)
    bid, ask, spread = get_bid_ask(ticker, timestamp)
    if bid == 0 or ask == 0:
        bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
    if bid == 0 or ask == 0:
        doBreak = True
    buffer.append((bid + ask)/2)
    running_spread += spread
if doBreak:
    #continue
    pass
buffer.append(running_spread/len(DAILY_AVG_NS))

running_spread = 0
for i in range(len(DAILY_AVG_NS)):
    timestamp = int(last_open_week_ns + DAILY_AVG_NS[i])
    bid, ask, spread = get_bid_ask(ticker, timestamp)
    if bid == 0 or ask == 0:
        bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
    if bid == 0 or ask == 0:
        doBreak = True
    running_spread += spread
    last_close_week = (last_close_week + timedelta(seconds=60*60*8)) + timedelta(seconds=seconds_to_next_market_open((last_close_week + timedelta(seconds=60*60*8))))
    last_open_week_ns = (int(last_close_week.timestamp()) * 1000000000)
if doBreak:
    #continue
    pass
buffer.append(running_spread/len(DAILY_AVG_NS))

running_spread = 0
for i in range(len(DAILY_AVG_NS)):
    timestamp = int(last_open_month_ns + DAILY_AVG_NS[i])
    bid, ask, spread = get_bid_ask(ticker, timestamp)
    if bid == 0 or ask == 0:
        bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
    if bid == 0 or ask == 0:
        doBreak = True
    running_spread += spread
    last_close_month = (last_close_month + timedelta(seconds=60*60*8))+ timedelta(seconds=seconds_to_next_market_open((last_close_month + timedelta(seconds=60*60*8))))
    last_open_month_ns = (int(last_close_month.timestamp()) * 1000000000)
if doBreak:
    #continue
    pass
buffer.append(running_spread/len(DAILY_AVG_NS))

next_open_date = time_datetime + timedelta(seconds=seconds_to_next_market_open(time_datetime))
next_next_open_date = next_open_date + timedelta(seconds=60*60*8) + timedelta(seconds=seconds_to_next_market_open(next_open_date+timedelta(seconds=60*60*8)))
next_open_date_ns = int(next_open_date.timestamp()) * 1000000000
next_next_open_date_ns = int(next_next_open_date.timestamp()) * 1000000000

for slot in DAY_OF_RESULTS:
    timestamp = next_open_date_ns + slot
    print(timestamp)
    bid, ask, spread = get_bid_ask(ticker, timestamp)
    if bid == 0 or ask == 0:
        bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
    if bid == 0 or ask == 0:
        doBreak = True
    buffer.append(bid)
    buffer.append(ask)
if doBreak:
    #continue
    pass

for slot in NEXT_DAY_RESULTS:
    timestamp = next_next_open_date_ns + slot
    print(timestamp)
    bid, ask, spread = get_bid_ask(ticker, timestamp)
    if bid == 0 or ask == 0:
        bid, ask, spread = get_bid_ask(ticker, timestamp+NS_PER_MIN)
    if bid == 0 or ask == 0:
        doBreak = True
    buffer.append(bid)
    buffer.append(ask)
if doBreak:
    #continue
    pass


print(next_open_date)
print(time_datetime)
print(int(time_datetime.timestamp()) * 1000000000)
print(ticker)
print(buffer)

1723469460000000000
1723469580000000000
1723469700000000000
1723470000000000000
1723471200000000000
1723474800000000000
1723478400000000000
1723482000000000000
1723485600000000000
1723489200000000000
1723555860000000000
1723557600000000000
2024-08-12 09:30:00
2024-08-09 16:15:00
1723234500000000000
OXY
[57.885000000000005, 58.415, 58.325, 0.010000000000000378, 0.010000000000002748, 0.01333333333333305, 58.63, 58.67, 58.58, 58.61, 58.67, 58.7, 58.99, 59.01, 58.58, 58.6, 58.8, 58.81, 58.91, 58.92, 58.92, 58.93, 58.94, 58.95, 58.97, 58.99, 57.5, 57.51, 57.3, 57.31]


In [11]:
print(ticker)
print(time_datetime)

TEX
2024-09-10 07:12:00


In [20]:
def get_bid_ask(ticker, stamp):
    url = 'https://api.polygon.io/v3/quotes/' + ticker + "?timestamp.gte=" + str(stamp) + "&order=asc&limit=1&sort=timestamp&apiKey=iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk"
    res = requests.get(url)
    res = res.json()["results"][0]
    bid = res["bid_price"]
    ask = res["ask_price"]
    spread = ask - bid
    return bid, ask, spread
#1744302354000000000
print(get_bid_ask("TEX", timestamp))

{'status': 'ERROR', 'request_id': '8640153d23707fdd254aab1ba73cff4f', 'error': 'failed to parse timestamp query params: time string was not any valid format: 1.7258922e 18'}


KeyError: 'results'

In [57]:
from polygon.rest import RESTClient

client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")

quotes = []
for t in client.list_quotes(
	ticker="AAPL",
	timestamp="2024-09-10",
	order="asc",
	limit=10,
	sort="timestamp",
	):
    quotes.append(t)

print(quotes)

KeyboardInterrupt: 

In [92]:
import csv
from datetime import datetime
import calendar
import requests
import time
import ast
from polygon.rest import RESTClient

client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")

with open("sic_dic.txt", "r") as f:
    content = f.read().strip()
    sic_dic = ast.literal_eval(content)


#INPUT
#ticker,timestamp,category,ST_impact_sentiment,text

#OUTPUT
#ticker,timestamp,category,impact,sentimet,DOW,timeESTInSecondsAfter4PM,secToNextOpen
#secFromLastClose, marketCap, enterpriseVal, PERaito, PBRatio, trailingPEG1Y
#first2Sic, first3Sic, first4Sic
#priceLastClose, priceLast2DayClose, priceLastWeekClose, priceLastMonthClose, priceLast3MonthClose, priceLastYearClose
#volLastCloseAvg, volLast2DayCloseAvg, volLastWeekCloseAvg, volLastMonthCloseAvg, volLast3MonthCloseAvg, volLastYearCloseAvg
#transLastClose, transLast2DayClose, transLastWeekClose, transLastMonthClose, transLast3MonthClose, transLastYearClose
#volNextDay, volNext2Day, volNext3Day

with open('TitanData.csv', newline='') as csvfile:
  quoter = list(csv.reader(csvfile, delimiter=','))
  for i in range(len(quoter)):
    try:
        row = quoter[i]
        buffer = row
        time_string = row[1]
        time_datetime = datetime.fromisoformat(time_string)
        
        if int(seconds_to_next_market_open(time_datetime)) != 0:
            with open('CleanedTitanData.csv', 'a', newline='') as csvfilewrite:
            #print(quoter[i][0] + " YAY")
                cleaner = csv.writer(csvfilewrite, delimiter=',')
                cleaner.writerow(buffer)
                csvfilewrite.close()
    except:
       print("WOAH FAILED SOME REASON")


In [52]:
from polygon.rest import RESTClient

client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")


print(time_datetime)
next_open_date = time_datetime + timedelta(seconds=seconds_to_next_market_open(time_datetime))
next_week_next_open_date = next_open_date + timedelta(days=7)
next_week_next_open_date = str(next_week_next_open_date)[:10]
next_open_date = str(next_open_date)[:10]

print(next_open_date)
print(next_week_next_open_date)

aggs = []
for a in client.list_aggs(
    ticker,
    1,
    "day",
    next_open_date,
    next_week_next_open_date,
    adjusted="true",
    sort="asc",
    limit=5,
):
    aggs.append(a)

next_open_vol = aggs[0].volume
next_2day_open_vol = aggs[1].volume
next_3day_open_vol = aggs[2].volume

print(next_open_vol)
print(next_2day_open_vol)
print(next_3day_open_vol)

print(aggs)


2024-09-10 07:12:00
2024-09-10
2024-09-17
656058
680050
566566
[Agg(open=50.61, high=51.03, low=49.58, close=49.91, volume=656058, vwap=50.045, timestamp=1725940800000, transactions=11204, otc=None), Agg(open=49.73, high=50.55, low=48.11, close=50.21, volume=680050, vwap=49.7961, timestamp=1726027200000, transactions=13438, otc=None), Agg(open=50.6, high=51.1, low=49.72, close=50.7, volume=566566, vwap=50.5497, timestamp=1726113600000, transactions=11561, otc=None), Agg(open=51.65, high=52.13, low=51.1, close=51.42, volume=460315, vwap=51.4447, timestamp=1726200000000, transactions=10556, otc=None), Agg(open=51.71, high=53.41, low=51.71, close=53.13, volume=822365, vwap=52.9042, timestamp=1726459200000, transactions=14333, otc=None), Agg(open=54.01, high=55.49, low=53.865, close=55.49, volume=932317, vwap=54.965, timestamp=1726545600000, transactions=16819, otc=None)]


In [44]:
from polygon.rest import RESTClient

client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")


print(time_datetime)
last_close_date = time_datetime - timedelta(seconds=seconds_from_last_market_close(time_datetime))
year_ago_last_close_date = last_close_date - timedelta(days=380)
year_ago_last_close_date = str(year_ago_last_close_date)[:10]
last_close_date = str(last_close_date)[:10]

print(last_close_date)
print(year_ago_last_close_date)

aggs = []
for a in client.list_aggs(
    ticker,
    1,
    "day",
    year_ago_last_close_date,
    last_close_date,
    adjusted="true",
    sort="desc",
    limit=250,
):
    aggs.append(a)

# if len(aggs) < 253:
#     except:
#         print(ticker + " LESS THAN 253 DAYS")

last_close_price = aggs[0].close
last_2day_close_price = aggs[1].close
last_week_close_price = aggs[5].close
last_month_close_price = aggs[22].close
last_3month_close_price = aggs[66].close
last_year_close_price = aggs[252].close

last_close_trans = aggs[0].transactions
last_2day_close_trans = aggs[1].transactions
last_week_close_trans = aggs[5].transactions
last_month_close_trans = aggs[22].transactions
last_3month_close_trans = aggs[66].transactions
last_year_close_trans = aggs[252].transactions

last_close_vol, last_2day_close_vol, last_week_close_vol, last_month_close_vol, last_3month_close_vol, last_year_close_vol = 0,0,0,0,0,0
running_vol = 0
for i in range(253):
    running_vol += aggs[i].volume
    if i == 0:
        last_close_vol = running_vol/(i+1)
    elif i == 1:
        last_2day_close_vol = running_vol/(i+1)
    elif i == 5:
        last_week_close_vol = running_vol/(i+1)
    elif i == 22:
        last_month_close_vol = running_vol/(i+1)
    elif i == 66:
        last_3month_close_vol = running_vol/(i+1)
    elif i == 252:
        last_year_close_vol = running_vol/(i+1)
print(last_close_price)
print(last_2day_close_price)
print(last_week_close_price)
print(last_month_close_price)
print(last_3month_close_price)
print(last_year_close_price)

print(last_close_trans)
print(last_2day_close_trans)
print(last_week_close_trans)
print(last_month_close_trans)
print(last_3month_close_trans)
print(last_year_close_trans)

print(last_close_vol)
print(last_2day_close_vol)
print(last_week_close_vol)
print(last_month_close_vol)
print(last_3month_close_vol)
print(last_year_close_vol)

print(aggs[0])
print(len(aggs))


2024-09-10 07:12:00
2024-09-09
2023-08-26
220.91
220.82
229
209.82
194.35
177.56
945464
663965
594157
741724
575318
1210080
67179965.0
57801488.0
49870459.333333336
43950519.39130435
62544123.731343284
59669928.0513834
Agg(open=220.82, high=221.27, low=216.71, close=220.91, volume=67179965.0, vwap=219.3818, timestamp=1725854400000, transactions=945464, otc=None)
260


In [48]:
import requests
ticker = "AZTA"
time_string = "2025-03-05"
time_datetime = datetime.fromisoformat(time_string)


headers = {
    'Content-Type': 'application/json',
}
endDate = time_string[:10]
startDate = str(time_datetime - timedelta(days=4))[:10]
url_string = "https://api.tiingo.com/tiingo/fundamentals/" + ticker.lower() + "/daily?token=c7dd3b16d9e5b5a440522b13fc9b527e45b07411"
requestResponse = requests.get(url_string, headers=headers, 
  params={'startDate' : startDate,
    "endDate" : endDate})
print(requestResponse.json()[-1])


2025-03-05
2025-03-01
{'date': '2025-03-05T00:00:00.000Z', 'marketCap': 2251825112.64, 'enterpriseVal': 1840641112.64, 'peRatio': -13.9185412374, 'pbRatio': 1.309988163, 'trailingPEG1Y': -3.8971915465}


In [112]:
from polygon.rest import RESTClient
client = RESTClient("iNnnzaGV1s5gWsE1aHbyoJWwa4H8RLbk")

financials = []
for f in client.vx.list_stock_financials(
	ticker="AAPL",
	order="asc",
	limit="1",
	sort="filing_date",
	):
    financials.append(f)

print(financials[-1])

StockFinancial(cik='0000320193', company_name='Apple Inc.', end_date='2024-12-28', filing_date='2025-01-31', financials=Financials(balance_sheet=BalanceSheet(assets=DataPoint(label='Assets', order=100, unit='USD', value=344085000000.0, derived_from=None, formula=None, source=None, xpath=None), current_assets=DataPoint(label='Current Assets', order=200, unit='USD', value=133240000000.0, derived_from=None, formula=None, source=None, xpath=None), cash=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), accounts_receivable=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), inventory=DataPoint(label='Inventory', order=230, unit='USD', value=6911000000.0, derived_from=None, formula=None, source=None, xpath=None), prepaid_expenses=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), other_current_assets=

In [79]:
import yfinance as yf

tickerdata = yf.Ticker('TSLA') #the tickersymbol for Tesla
print(tickerdata.info.get['sector'])

429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/TSLA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=TSLA&crumb=Edge%3A+Too+Many+Requests


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [1]:
import requests
import json

url = f'https://eodhd.com/api/fundamentals/AAPL.US?api_token=demo&fmt=json'
data = requests.get(url).json()

#print(data)

print(data["Highlights"].keys())

dict_keys(['MarketCapitalization', 'MarketCapitalizationMln', 'EBITDA', 'PERatio', 'PEGRatio', 'WallStreetTargetPrice', 'BookValue', 'DividendShare', 'DividendYield', 'EarningsShare', 'EPSEstimateCurrentYear', 'EPSEstimateNextYear', 'EPSEstimateNextQuarter', 'EPSEstimateCurrentQuarter', 'MostRecentQuarter', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'RevenueTTM', 'RevenuePerShareTTM', 'QuarterlyRevenueGrowthYOY', 'GrossProfitTTM', 'DilutedEpsTTM', 'QuarterlyEarningsGrowthYOY'])


In [102]:
import pandas as pd

# Load CSV into a DataFrame
df = pd.read_csv("TitanData.csv")

# Select row by index and get unique valuesw
unique_values = df["ticker"].unique()

print(len(unique_values))

with open("output.txt", "w") as f:
    f.write(", ".join(map(str, unique_values)))

3604


In [104]:
## Get sector from polygon

with open("output.txt", "r") as f:
    content = f.read()
    inner_list = [elt.strip() for elt in content.split(',')]


print(len(inner_list))

3604


In [14]:
import ast

with open("sic_dic.txt", "r") as f:
    content = f.read().strip()
    my_dict = ast.literal_eval(content)

print(my_dict["A"])

3826


In [12]:
#for item in requestResponse:
    #print(item)


ticker_to_sic = {item["ticker"].upper(): item["sicCode"] for item in requestResponse.json()}
print(ticker_to_sic)

with open("SIC_DIC.txt", "w") as f:
    f.write(str(ticker_to_sic))

{'A': 3826, 'AA': 3334, 'AAAB': 6022, 'AAAGY': 2834, 'AAAP': 2834, 'AABC': 6035, 'AAC': 6770, 'AAC-U': None, 'AAC1': 6153, 'AACB': 6770, 'AACBU': 6770, 'AACC': 6153, 'AACE': 6099, 'AACG': 8200, 'AACH': 8093, 'AACI': 6770, 'AACIU': 6770, 'AACPF': 6770, 'AACT': 6770, 'AACT-U': 6770, 'AADV': 6036, 'AAGR': 100, 'AAH': 6798, 'AAI': 4512, 'AAI1': 8711, 'AAIC': 6798, 'AAII': 8734, 'AAIIQ': 8734, 'AAIR': 4522, 'AAL': 4512, 'AAM': 6770, 'AAM-U': 6770, 'AAMC': 6500, 'AAMCF': 6500, 'AAME': 6311, 'AAMI': 6282, 'AAMRQ': 4512, 'AAN': 7359, 'AANB': 6021, 'AAOI': 3674, 'AAON': 3585, 'AAP': 5531, 'AAPC': 7990, 'AAPG': 2834, 'AAPL': 3571, 'AAQC': 6770, 'AAQC-U': None, 'AAQL': 2834, 'AARD': 2834, 'AAT': 6798, 'AATC': 3829, 'AATI': 3674, 'AATI1': 8711, 'AATT': 3679, 'AAUAF': 1000, 'AAV': 1311, 'AAV1': 6500, 'AAVVF': 1311, 'AAWHQ': 4522, 'AAWW': 4522, 'AB': 6282, 'ABACF': 1311, 'ABACQ': 1311, 'ABAN': 6022, 'ABAT': 1000, 'ABAX': 3829, 'ABBC': 6036, 'ABBK': 6036, 'ABBNY': 3613, 'ABBV': 2834, 'ABCB': 6022, 'A

In [3]:
import requests
headers = {
    'Content-Type': 'application/json'
}
requestResponse = requests.get("https://api.tiingo.com/tiingo/fundamentals/meta?token=c7dd3b16d9e5b5a440522b13fc9b527e45b07411", headers=headers)
print(requestResponse.json())